# Chatbot POC - EcoMarket Retail

Versión mejorada del prototipo con etiquetas menos solapadas, threshold único, reglas de seguridad, consultas a inventario/pedidos simulados y memoria básica de conversación.

In [5]:
from dataclasses import dataclass
import csv
from datetime import datetime
from pathlib import Path
import re
from typing import Optional
import unicodedata

import pandas as pd
from transformers import pipeline


# =====================================================================
# 1. CONFIGURACION GENERAL
# =====================================================================
MODEL_NAME = 'MoritzLaurer/mDeBERTa-v3-base-mnli-xnli'
CONFIDENCE_THRESHOLD = 0.55
INTENCION_SOPORTE = 'Hablar con soporte humano'
RUTA_DATOS_REENTRENAMIENTO = Path("C:\\data_sciences\\GIT\\ChatBot_EcoMarket\\ChatBot_EcoMarket\\data\\preguntas_clientes_reentrenamiento_1.0.3.csv")
RUTA_INVENTARIO_PRODUCTOS = Path("C:\\data_sciences\\GIT\\ChatBot_EcoMarket\\ChatBot_EcoMarket\\data\\inventario_productos.xlsx")
RUTA_PEDIDOS = Path("C:\\data_sciences\\GIT\\ChatBot_EcoMarket\\ChatBot_EcoMarket\\data\\pedidos.xlsx")
COMANDOS_SALIDA = {'salir', 'escape', 'exit', 'quit', 'cerrar'}
CANTIDAD_POR_DEFECTO = 1
CAMPOS_REENTRENAMIENTO = [
    'fecha_hora',
    'pregunta_cliente',
    'intencion_detectada',
    'intencion_correcta',
    'confianza',
    'origen_intencion',
    'respuesta_bot',
    'respuesta_correcta',
    'aprobado_para_entrenamiento',
    'notas',
]

ETIQUETAS_NEGOCIO = [
    'Estado del pedido',
    'Incidencia con pedido incompleto o no recibido',
    'Devoluciones y cambios',
    'Producto dañado caducado o en mal estado',
    'Disponibilidad de productos',
    'Promociones y cupones',
    'Métodos de pago y compra',
    INTENCION_SOPORTE,
]


# =====================================================================
# 2. FUENTES DE DATOS DESDE EXCEL
# =====================================================================
def validar_columnas_excel(df, columnas_requeridas, nombre_archivo):
    columnas_faltantes = [columna for columna in columnas_requeridas if columna not in df.columns]
    if columnas_faltantes:
        raise ValueError(
            f"El archivo {nombre_archivo} no tiene las columnas requeridas: {', '.join(columnas_faltantes)}"
        )


def cargar_catalogo_productos(ruta_excel):
    df = pd.read_excel(ruta_excel)
    validar_columnas_excel(df, ['producto', 'stock', 'aliases'], ruta_excel.name)

    if 'activo' in df.columns:
        df = df[df['activo'].fillna('si').astype(str).str.lower().str.strip().isin(['si', 'sí', 'true', '1'])]

    catalogo = {}
    for _, fila in df.iterrows():
        producto = str(fila['producto']).strip()
        if not producto:
            continue

        aliases = [
            alias.strip()
            for alias in str(fila['aliases']).split('|')
            if alias.strip()
        ]

        if producto not in aliases:
            aliases.insert(0, producto)

        catalogo[producto] = {
            'stock': int(fila['stock']),
            'aliases': aliases,
        }

    return catalogo


def cargar_base_pedidos(ruta_excel):
    df = pd.read_excel(ruta_excel)
    validar_columnas_excel(df, ['id_pedido', 'estado', 'fecha_entrega'], ruta_excel.name)

    pedidos = {}
    for _, fila in df.iterrows():
        id_pedido = str(fila['id_pedido']).strip().upper()
        if not id_pedido:
            continue

        pedidos[id_pedido] = {
            'estado': str(fila['estado']).strip(),
            'fecha_entrega': str(fila['fecha_entrega']).strip(),
        }

    return pedidos


CATALOGO_PRODUCTOS = cargar_catalogo_productos(RUTA_INVENTARIO_PRODUCTOS)
BASE_DATOS_PEDIDOS = cargar_base_pedidos(RUTA_PEDIDOS)

POLITICAS = {
    'devoluciones': (
        'Las devoluciones pueden solicitarse hasta 30 días después de la compra, '
        'presentando ticket o número de pedido. En productos frescos, perecederos '
        'o de higiene pueden aplicar restricciones.'
    ),
    'promociones': (
        'Las promociones y cupones dependen de sus condiciones. Algunas ofertas '
        'pueden ser exclusivas de tienda física o del e-commerce.'
    ),
    'pagos': (
        'Aceptamos tarjeta de crédito, tarjeta de débito y los métodos disponibles '
        'durante el checkout online.'
    ),
}


@dataclass
class EstadoConversacion:
    esperando_id_pedido: bool = False
    ultima_intencion: Optional[str] = None


# =====================================================================
# 3. UTILIDADES
# =====================================================================
def normalizar_texto(texto):
    texto = texto.lower().strip()
    texto = unicodedata.normalize('NFD', texto)
    return ''.join(c for c in texto if unicodedata.category(c) != 'Mn')


def extraer_id_pedido(mensaje_usuario):
    match = re.search(r'\bped[-\s]?\d+\b', mensaje_usuario, flags=re.IGNORECASE)
    if not match:
        return None
    return match.group().replace(' ', '-').upper()


def extraer_cantidad_solicitada(mensaje_usuario):
    mensaje_normalizado = normalizar_texto(mensaje_usuario)
    patrones = [
        r'\b(?:quiero|necesito|comprar|llevar|pedir|solicito)\s+(\d+)\b',
        r'\b(\d+)\s+(?:unidades|uds|piezas|kilos|kg|paquetes|productos)?\b',
    ]

    for patron in patrones:
        match = re.search(patron, mensaje_normalizado)
        if match:
            cantidad = int(match.group(1))
            if cantidad > 0:
                return cantidad

    return CANTIDAD_POR_DEFECTO


def buscar_producto(mensaje_usuario):
    mensaje_normalizado = normalizar_texto(mensaje_usuario)
    for producto, datos in CATALOGO_PRODUCTOS.items():
        for alias in datos['aliases']:
            if normalizar_texto(alias) in mensaje_normalizado:
                return producto
    return None


def contiene_alguna(mensaje_normalizado, expresiones):
    return any(expresion in mensaje_normalizado for expresion in expresiones)


def necesita_soporte_por_palabras_clave(mensaje_usuario):
    mensaje_normalizado = normalizar_texto(mensaje_usuario)
    palabras_clave = [
        'humano', 'agente', 'persona', 'reclamo', 'queja', 'urgente',
        'denuncia', 'no me ayudan'
    ]
    return contiene_alguna(mensaje_normalizado, palabras_clave)


def detectar_intencion_por_reglas(mensaje_usuario):
    mensaje_normalizado = normalizar_texto(mensaje_usuario)

    if necesita_soporte_por_palabras_clave(mensaje_usuario):
        return INTENCION_SOPORTE

    if contiene_alguna(mensaje_normalizado, ['caduc', 'vencid', 'mal estado', 'danad', 'roto', 'podrid', 'contaminad']):
        return 'Producto dañado caducado o en mal estado'

    if contiene_alguna(mensaje_normalizado, [
        'no ha llegado', 'no llego', 'no recibi', 'incompleto', 'faltan',
        'falta', 'equivocado', 'producto incorrecto', 'entrega tarde', 'retraso'
    ]):
        return 'Incidencia con pedido incompleto o no recibido'

    if contiene_alguna(mensaje_normalizado, ['devolver', 'devolucion', 'cambiar', 'cambio', 'reembolso']):
        return 'Devoluciones y cambios'

    if contiene_alguna(mensaje_normalizado, ['cupon', 'promocion', 'descuento', 'oferta', 'puntos']):
        return 'Promociones y cupones'

    if contiene_alguna(mensaje_normalizado, ['tarjeta', 'pago', 'pagar', 'checkout', 'paypal', 'bizum']):
        return 'Métodos de pago y compra'

    if buscar_producto(mensaje_usuario) and (
        contiene_alguna(mensaje_normalizado, ['disponible', 'stock', 'inventario', 'tienen', 'hay', 'comprar','disponibilidad', 'quiero', 'necesito', 'llevar', 'pedir'])
        or extraer_cantidad_solicitada(mensaje_usuario) > CANTIDAD_POR_DEFECTO
    ):
        return 'Disponibilidad de productos'

    if extraer_id_pedido(mensaje_usuario) or contiene_alguna(mensaje_normalizado, ['estado del pedido', 'donde esta mi pedido', 'seguimiento del pedido']):
        return 'Estado del pedido'

    return None


# =====================================================================
# 4. CARGA DEL MODELO
# =====================================================================
print('Cargando el modelo de NLP... espera unos segundos')
clasificador = pipeline('zero-shot-classification', model=MODEL_NAME)


# =====================================================================
# 5. FUNCIONES PRINCIPALES
# =====================================================================
def procesar_mensaje(mensaje_usuario):
    intencion_por_regla = detectar_intencion_por_reglas(mensaje_usuario)
    if intencion_por_regla:
        return intencion_por_regla, 1.0, [(intencion_por_regla, 1.0)], 'regla'

    resultado = clasificador(mensaje_usuario, ETIQUETAS_NEGOCIO)
    ranking = list(zip(resultado['labels'], resultado['scores']))
    intencion_detectada = resultado['labels'][0]
    score = resultado['scores'][0]

    if score < CONFIDENCE_THRESHOLD:
        return INTENCION_SOPORTE, score, ranking, 'modelo_zero_shot_baja_confianza'

    return intencion_detectada, score, ranking, 'modelo_zero_shot'


def consultar_estado_pedido(mensaje_usuario, estado):
    id_pedido = extraer_id_pedido(mensaje_usuario)

    if not id_pedido:
        estado.esperando_id_pedido = True
        return 'Claro, puedo revisar tu pedido. Indícame la referencia, por ejemplo PED-123.'

    estado.esperando_id_pedido = False

    if id_pedido not in BASE_DATOS_PEDIDOS:
        return f'No encuentro el pedido {id_pedido}. Revisa la referencia o te derivo con soporte.'

    info = BASE_DATOS_PEDIDOS[id_pedido]
    estado_pedido = info['estado']
    fecha_entrega = info['fecha_entrega']
    return f'Tu pedido {id_pedido} se encuentra en estado: {estado_pedido}. Entrega estimada: {fecha_entrega}.'


def responder_disponibilidad_producto(mensaje_usuario):
    producto = buscar_producto(mensaje_usuario)

    if not producto:
        return 'Puedo consultar disponibilidad online, pero necesito el producto exacto. Por ejemplo: leche vegetal, tofu o pan sin gluten.'

    stock = CATALOGO_PRODUCTOS[producto]['stock']
    cantidad_solicitada = extraer_cantidad_solicitada(mensaje_usuario)

    if stock <= 0:
        return f'Ahora mismo {producto} aparece agotado en el inventario online. Puedes revisar más tarde o consultar alternativas.'

    if cantidad_solicitada > stock:
        return (
            f'Ahora mismo solo hay {stock} unidades de {producto} disponibles. '
            f'No puedo confirmar una compra de {cantidad_solicitada} unidades con el inventario actual. '
            'Puedes reducir la cantidad o consultar alternativas.'
        )

    if cantidad_solicitada > CANTIDAD_POR_DEFECTO:
        return (
            f'Sí, hay stock suficiente para {cantidad_solicitada} unidades de {producto}. '
            f'Actualmente figuran {stock} unidades en el inventario online. La disponibilidad puede variar al finalizar la compra.'
        )

    return f'Sí, actualmente figuran {stock} unidades de {producto} en el inventario online. La disponibilidad puede variar al finalizar la compra.'


def responder_incidencia_pedido(mensaje_usuario):
    id_pedido = extraer_id_pedido(mensaje_usuario)
    if id_pedido:
        return f'Lamento la incidencia con el pedido {id_pedido}. Dime si faltan productos, llegó tarde o hubo un error en la entrega.'
    return 'Lamento lo ocurrido con tu pedido. Indícame el número de pedido y si no llegó, llegó incompleto o contiene productos equivocados.'


def responder_producto_mal_estado(mensaje_usuario):
    producto = buscar_producto(mensaje_usuario)
    if producto:
        return f'Siento que hayas recibido {producto} en mal estado. Conserva el ticket o número de pedido y, si puedes, una foto. Te derivo con atención al cliente.'
    return 'Siento que hayas recibido un producto en mal estado. Indícame el producto, el número de pedido y si tienes foto o ticket.'


def obtener_respuesta(intencion, mensaje_usuario, estado):
    if estado.esperando_id_pedido and extraer_id_pedido(mensaje_usuario):
        return consultar_estado_pedido(mensaje_usuario, estado)

    estado.ultima_intencion = intencion

    if intencion == 'Estado del pedido':
        return consultar_estado_pedido(mensaje_usuario, estado)
    if intencion == 'Incidencia con pedido incompleto o no recibido':
        return responder_incidencia_pedido(mensaje_usuario)
    if intencion == 'Disponibilidad de productos':
        return responder_disponibilidad_producto(mensaje_usuario)
    if intencion == 'Producto dañado caducado o en mal estado':
        return responder_producto_mal_estado(mensaje_usuario)
    if intencion == 'Devoluciones y cambios':
        return POLITICAS['devoluciones'] + ' Si quieres, puedo ayudarte a preparar la solicitud.'
    if intencion == 'Promociones y cupones':
        return POLITICAS['promociones'] + ' Si un cupón no funciona, revisa fecha de validez, importe mínimo y canal de compra.'
    if intencion == 'Métodos de pago y compra':
        return POLITICAS['pagos']
    if intencion == INTENCION_SOPORTE:
        return 'Quiero evitar darte una respuesta incorrecta. Te puedo derivar con atención al cliente; antes, cuéntame brevemente qué ocurrió.'

    return 'No he entendido bien tu consulta. Puedo ayudarte con pedidos, devoluciones, productos, promociones o soporte.'


Cargando el modelo de NLP... espera unos segundos


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
# =====================================================================
# 6. TEST SUITE AUTOMATICO
# =====================================================================
def ejecutar_tests():
    test_queries = [
        'Quiero devolver un producto que compré online',
        'Mi pedido no ha llegado todavía',
        'Me faltan productos en la compra',
        'Tengo un cupón y no me funciona',
        'Recibí un yogur caducado',
        'Quiero saber si tienen leche vegetal disponible',
        'Quiero comprar 100 manzanas',
        'Necesito 6 panes sin gluten',
        '¿Dónde está mi pedido PED-123?',
        '¿Aceptan tarjeta de crédito?',
        'Quiero hablar con una persona',
    ]

    estado = EstadoConversacion()
    print('\n' + '=' * 60)
    print('--- INICIANDO TEST SUITE AUTOMATICO (EPIC 4) ---')
    print('=' * 60)

    for query in test_queries:
        intencion, score, ranking, origen = procesar_mensaje(query)
        respuesta_bot = obtener_respuesta(intencion, query, estado)

        print(f'Frase: {query}')
        print(f'Detectado: {intencion} (Confianza: {score:.2f} | Origen: {origen})')
        print('Top 3 intenciones:')
        for label, label_score in ranking[:3]:
            print(f'  - {label}: {label_score:.2f}')
        print(f'Accion Bot: {respuesta_bot}')
        print('-' * 60)


ejecutar_tests()



--- INICIANDO TEST SUITE AUTOMATICO (EPIC 4) ---
Frase: Quiero devolver un producto que compré online
Detectado: Devoluciones y cambios (Confianza: 1.00)
Top 3 intenciones:
  - Devoluciones y cambios: 1.00
Accion Bot: Las devoluciones pueden solicitarse hasta 30 días después de la compra, presentando ticket o número de pedido. En productos frescos, perecederos o de higiene pueden aplicar restricciones. Si quieres, puedo ayudarte a preparar la solicitud.
------------------------------------------------------------
Frase: Mi pedido no ha llegado todavía
Detectado: Incidencia con pedido incompleto o no recibido (Confianza: 1.00)
Top 3 intenciones:
  - Incidencia con pedido incompleto o no recibido: 1.00
Accion Bot: Lamento lo ocurrido con tu pedido. Indícame el número de pedido y si no llegó, llegó incompleto o contiene productos equivocados.
------------------------------------------------------------
Frase: Me faltan productos en la compra
Detectado: Incidencia con pedido incompleto o 

In [3]:
# =====================================================================
# 7. CHAT INTERACTIVO
# =====================================================================
def asegurar_csv_reentrenamiento():
    if not RUTA_DATOS_REENTRENAMIENTO.exists():
        RUTA_DATOS_REENTRENAMIENTO.parent.mkdir(parents=True, exist_ok=True)
        with RUTA_DATOS_REENTRENAMIENTO.open('w', newline='', encoding='utf-8') as archivo:
            writer = csv.DictWriter(archivo, fieldnames=CAMPOS_REENTRENAMIENTO)
            writer.writeheader()
        return

    with RUTA_DATOS_REENTRENAMIENTO.open('r', newline='', encoding='utf-8') as archivo:
        reader = csv.DictReader(archivo)
        filas = list(reader)
        campos_actuales = reader.fieldnames or []

    if campos_actuales == CAMPOS_REENTRENAMIENTO:
        return

    for fila in filas:
        fila.setdefault('intencion_correcta', '')
        fila.setdefault('respuesta_correcta', '')
        fila.setdefault('origen_intencion', 'desconocido')
        fila.setdefault('aprobado_para_entrenamiento', 'no')
        fila.setdefault('notas', '')

    with RUTA_DATOS_REENTRENAMIENTO.open('w', newline='', encoding='utf-8') as archivo:
        writer = csv.DictWriter(archivo, fieldnames=CAMPOS_REENTRENAMIENTO)
        writer.writeheader()
        for fila in filas:
            writer.writerow({campo: fila.get(campo, '') for campo in CAMPOS_REENTRENAMIENTO})


def guardar_pregunta_reentrenamiento(pregunta, intencion, confianza, respuesta, origen_intencion):
    asegurar_csv_reentrenamiento()

    with RUTA_DATOS_REENTRENAMIENTO.open('a', newline='', encoding='utf-8') as archivo:
        writer = csv.DictWriter(archivo, fieldnames=CAMPOS_REENTRENAMIENTO)
        writer.writerow({
            'fecha_hora': datetime.now().isoformat(timespec='seconds'),
            'pregunta_cliente': pregunta,
            'intencion_detectada': intencion,
            'intencion_correcta': '',
            'confianza': round(confianza, 4),
            'origen_intencion': origen_intencion,
            'respuesta_bot': respuesta,
            'respuesta_correcta': '',
            'aprobado_para_entrenamiento': 'no',
            'notas': '',
        })


def ejecutar_chat():
    estado = EstadoConversacion()

    print('\n' + '=' * 60)
    print('--- CHATBOT INTERACTIVO ECOMARKET RETAIL ---')
    print('=' * 60)
    print("Escribe 'salir' para terminar.\n")

    while True:
        texto = input('Usuario: ').strip()

        if normalizar_texto(texto) in COMANDOS_SALIDA:
            print('Bot: Hasta pronto. Cerrando sistema.')
            break

        if not texto:
            continue

        intencion, score, _, origen = procesar_mensaje(texto)
        respuesta_conversacional = obtener_respuesta(intencion, texto, estado)
        guardar_pregunta_reentrenamiento(texto, intencion, score, respuesta_conversacional, origen)

        print(f'Bot: {respuesta_conversacional}')
        print(f'Intencion: {intencion} | Confianza: {score:.2f} | Origen: {origen}')
        print('-' * 60)


# Ejecuta esta linea cuando quieras iniciar la demo interactiva:
ejecutar_chat()



--- CHATBOT INTERACTIVO ECOMARKET RETAIL ---
Escribe 'salir' para terminar.

Bot: Lamento lo ocurrido con tu pedido. Indícame el número de pedido y si no llegó, llegó incompleto o contiene productos equivocados.
Intencion: Incidencia con pedido incompleto o no recibido | Confianza: 1.00
------------------------------------------------------------
Bot: Quiero evitar darte una respuesta incorrecta. Te puedo derivar con atención al cliente; antes, cuéntame brevemente qué ocurrió.
Intencion: Hablar con soporte humano | Confianza: 0.48
------------------------------------------------------------
Bot: Hasta pronto. Cerrando sistema.
